# End-to-end toy project: predicting placement

Run each cell in order with **Shift + Enter**. Each step matches a section of the Note.
The data (`data/placement.csv`) is the toy dataset from the course: 100 students' CGPA, IQ and placement.

## 1. Load and look at the data

In [ ]:
import numpy as np
import pandas as pd

df = pd.read_csv("data/placement.csv")   # read the table from a CSV file
print(df.shape)                           # (rows, columns)
df.head()                                 # the first 5 rows

In [ ]:
df.info()                                 # column types and non-missing counts: no missing values here

## 2. Clean: drop the unneeded index column

In [ ]:
df = df.iloc[:, 1:]                       # all rows, every column from position 1 onwards
df.head()

## 3. Explore: plot CGPA against IQ, coloured by placement

In [ ]:
import plotly.express as px

px.scatter(df, x="cgpa", y="iq", color=df["placement"].map({1: "placed", 0: "not placed"}),
           labels={"color": ""}, template="simple_white").show()

## 4. Split inputs (X) from output (y)

In [ ]:
X = df.iloc[:, 0:2]      # cgpa and iq: the inputs (independent variables)
y = df.iloc[:, -1]       # placement: the output (dependent variable)
print(X.shape, y.shape)

## 5. Split into training and test sets

`test_size=0.1` keeps 10% of the rows (10 students) hidden for testing.
`random_state=1` fixes the random shuffle, so we get the same split every time we run this.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=1)
print(X_train.shape, X_test.shape)
X_test

## 6. Scale the inputs

The scaler learns each column's mean and spread **from the training set only** (`fit_transform`),
then applies the same change to the test set (`transform`).

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # learn mean and spread, then scale
X_test_scaled = scaler.transform(X_test)         # scale with the training set's numbers
print(X_train_scaled[:5].round(2))

## 7. Train the model

In [ ]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression()
clf.fit(X_train_scaled, y_train)                 # training: find the best boundary

## 8. Evaluate on the test set

In [ ]:
from sklearn.metrics import accuracy_score

y_pred = clf.predict(X_test_scaled)
print("predicted:", y_pred)
print("actual:   ", y_test.values)
print("accuracy: ", accuracy_score(y_test, y_pred))

## 9. Save the model (and the scaler!)

The model only understands **scaled** inputs, so the scaler must be saved with it.
`pickle` turns Python objects into a file that another program can load.

In [ ]:
import pickle

with open("model.pkl", "wb") as f:               # "wb" = write in binary mode
    pickle.dump({"scaler": scaler, "model": clf}, f)
print("saved model.pkl")

## 10. A small website: load the saved file and predict

This is the deployment step, run on our own machine with Dash. Type an IQ and a CGPA and press **Predict**.

In [ ]:
import pickle
from dash import Dash, dcc, html, Input, Output, State

with open("model.pkl", "rb") as f:               # "rb" = read in binary mode
    saved = pickle.load(f)

app = Dash(__name__)
app.layout = html.Div([
    html.H3("Will this student be placed?"),
    html.Label("IQ "), dcc.Input(id="iq", type="number", value=123),
    html.Label("  CGPA "), dcc.Input(id="cgpa", type="number", value=6.8, step=0.1),
    html.Button("Predict", id="go"),
    html.H4(id="answer"),
])

@app.callback(Output("answer", "children"), Input("go", "n_clicks"), State("iq", "value"), State("cgpa", "value"))
def predict(_, iq, cgpa):
    row = pd.DataFrame({"cgpa": [cgpa], "iq": [iq]})            # same columns as in training
    placed = saved["model"].predict(saved["scaler"].transform(row))[0]
    return "Placed" if placed == 1 else "Not placed"

app.run(jupyter_mode="inline", port=8057)